In [2]:
import torch
import torch.nn as nn
from config import *
from src.datasets.dataset import get_data_loader
from src.models.models import ResNet50Model
from src.engine.trainer import Trainer
from src.engine.predictor import Predictor
from src.engine.checkpoint_manager import CheckpointManager
from src.utils.freeze import freeze, unfreeze
from src.utils.device import get_device

ModuleNotFoundError: No module named 'config'

In [4]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Failed to issue request POST https://colab.research.google.com/tun/m/credentials-propagation/m-s-kkb-ass1c1-3v1ztr4zw91f0?authtype=dfs_ephemeral&version=2&dryrun=false&propagate=true&record=false&authuser=0: Bad Request
Response body: 
<!DOCTYPE html>
<html lang=en>
  <meta charset=utf-8>
  <meta name=viewport content="initial-scale=1, minimum-scale=1, width=device-width">
  <title>Error 400 (Bad Request)!!1</title>
  <style>
    *{margin:0;padding:0}html,code{font:15px/22px arial,sans-serif}html{background:#fff;color:#222;padding:15px}body{margin:7% auto 0;max-width:390px;min-height:180px;padding:30px 0 15px}* > body{background:url(//www.google.com/images/errors/robot.png) 100% 5px no-repeat;padding-right:205px}p{margin:11px 0 22px;overflow:hidden}ins{color:#777;text-decoration:none}a img{border:0}@media screen and (max-width:772px){body{background:none;margin-top:0;max-width:none;padding-right:0}}#logo{background:url(//www.google.com/images/logos/errorpage/error_logo-15

In [2]:
checkpoint_manager = CheckpointManager ("checkpoints")
device = get_device ()
print (device)

mps


In [3]:
model = ResNet50Model (True, NUM_CLASSES)
print (model)

ResNet50Model(
  (backbone): ResNet50Backbone(
    (blocks): ModuleList(
      (0): Sequential(
        (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      )
      (1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats

/Users/mac/Library/CloudStorage/OneDrive-ctu.edu.vn/myworkspace/celeb_faces_regconigtion/.venv/lib/python3.12/site-packages/torch/nn/modules/lazy.py:181: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


In [4]:
freeze (model.backbone)

In [5]:
optimizer = torch.optim.Adam (filter (lambda p: p.requires_grad, model.parameters ()), lr = LR)

In [6]:
criterion = nn.CrossEntropyLoss ()

In [7]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau (optimizer, mode = "max", factor = 0.1, patience = 3)

In [8]:
train_loader, val_loader, test_loader = get_data_loader (DATASET_ROOT, BATCH_SIZE, True)

In [9]:
trainer = Trainer (model, train_loader, val_loader, criterion, optimizer, device, scheduler)

In [10]:
checkpoint = checkpoint_manager.load ("resnet50.pth", model, optimizer, scheduler, device)

In [11]:
trainer.fit (0, 5)

Epoch 0/5
Loss: 2.7378
Val Acc: 0.2965
Epoch 1/5
Loss: 2.0665
Val Acc: 0.4602
Epoch 2/5
Loss: 1.7155
Val Acc: 0.5133
Epoch 3/5
Loss: 1.4647
Val Acc: 0.5177
Epoch 4/5
Loss: 1.2945
Val Acc: 0.5752


ResNet50Model(
  (backbone): ResNet50Backbone(
    (blocks): ModuleList(
      (0): Sequential(
        (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      )
      (1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats

In [12]:
checkpoint_manager.save ("resnet50.pth",model, optimizer, scheduler, 5)

TypeError: save() missing 1 required positional argument: 'f'

sample_data
